# 01 — Data Cleaning & Quality Audit

**Goal:** audit the raw hospital admissions export, document every data-quality issue found, justify each cleaning decision, and produce the processed dataset used by all downstream analysis.

**Why this matters:** in healthcare analytics, silent data errors (duplicate registrations, kiosk export bugs) directly distort KPIs like wait time and readmission rate. Every fix below is documented so the pipeline is auditable.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

# Make the shared cleaning module importable from the project root
sys.path.append(str(Path.cwd().parent))

raw = pd.read_csv('../data/raw/hospital_admissions.csv')
print(f'Raw shape: {raw.shape}')
raw.head()

## 1. Quality audit

Before touching anything, quantify the problems. Five issue types were found in the raw export.

In [ ]:
print('Exact duplicate rows :', raw.duplicated().sum())
print('Gender encodings     :', sorted(raw['gender'].dropna().unique()))
print('Missing wait times   :', raw['wait_time_minutes'].isna().sum())
print('Missing insurance    :', raw['insurance_type'].isna().sum())
print('Negative wait times  :', (raw['wait_time_minutes'] < 0).sum())

# Mixed date formats: legacy system exported DD/MM/YYYY for some rows
legacy_dates = raw['admission_date'].str.contains('/', na=False).sum()
print('Legacy-format dates  :', legacy_dates)

## 2. Cleaning decisions

| Issue | Decision | Rationale |
|---|---|---|
| Duplicate rows | Drop exact duplicates | Double-submitted registration forms, not separate visits |
| Gender as `M`/`F`/`Male`/`Female` | Map to `Male`/`Female` | Old EHR used abbreviations; standardise for grouping |
| Negative wait times | Set to NaN, then impute | Kiosk export bug; values are not real measurements |
| Missing wait times | Impute with **department** median | Waits differ hugely by department (Emergency vs Oncology); a global median would bias both |
| Missing insurance | Keep as `Unknown` category | Billing follow-up needs these records visible, not hidden by imputation |
| Mixed date formats | Two-pass parse (ISO, then DD/MM/YYYY) | Naive parsing would silently swap day and month |

In [ ]:
from scripts.clean_data import clean, PROCESSED_PATH

df = clean(raw)

# Resolve output path relative to the project root, not the notebooks dir
out_path = Path('..') / PROCESSED_PATH
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print(f'Cleaned shape: {df.shape}  (removed {len(raw) - len(df)} duplicates)')

## 3. Validation

A cleaning step is only done when it can be proven done.

In [ ]:
assert df.duplicated().sum() == 0, 'duplicates remain'
assert set(df['gender'].unique()) <= {'Male', 'Female'}, 'gender not standardised'
assert (df['wait_time_minutes'] >= 0).all(), 'negative waits remain'
assert df['admission_date'].isna().sum() == 0, 'unparsed dates remain'
assert df['insurance_type'].isna().sum() == 0, 'missing insurance remains'

print('All validation checks passed.')
df[['admission_date', 'admission_month', 'age_group', 'discharge_date']].head()